# <b>Camera Run</b>

In [ ]:
from picamera2 import Picamera2, Preview
import cv2
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

# 카메라 객체 생성
picam2 = Picamera2()
camera_config = picam2.create_preview_configuration(
    main={"size": (320, 180), "format": "BGR888"}
)
picam2.configure(camera_config)
picam2.start()

# 비디오 스트림을 위한 위젯
video_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='320px', height='180px'))
display(video_widget)

def convert_to_bytes(image):
    _, buffer = cv2.imencode('.jpg', image)
    return buffer.tobytes()

while True:
    frame = picam2.capture_array()
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

    # JPEG 형식으로 인코딩
    video_widget.value = convert_to_bytes(frame)

    # 이미지 업데이트를 5 FPS로 제한
    time.sleep(0.2)

clear_output(wait=True)


# <b>HSV Color Trackbar</b>

In [ ]:
from picamera2 import Picamera2, Preview
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import threading
import time

# 전역 변수
picam2 = Picamera2()
camera_config = picam2.create_preview_configuration(
    main={"size": (320, 180), "format": "BGR888"}
)
picam2.configure(camera_config)
picam2.start()

def find_contours_in_color_range(image, lower_hsv, upper_hsv):
    """지정된 색상 범위 내에서 윤곽선을 찾고 그리기"""
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    color_mask = cv2.inRange(hsv_image, lower_hsv, upper_hsv)
    masked_image = cv2.bitwise_and(image, image, mask=color_mask)
    gray_mask = cv2.cvtColor(masked_image, cv2.COLOR_BGR2GRAY)
    _, binary_mask = cv2.threshold(gray_mask, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(masked_image, contours, -1, (0, 255, 0), 2)  # 윤곽선을 마스크된 이미지에 그림
    return image, color_mask, masked_image

def convert_to_bytes(image):
    """OpenCV 이미지를 JPEG 바이트 배열로 변환"""
    _, buffer = cv2.imencode('.jpg', image)
    return buffer.tobytes()

def update_image(*args):
    """슬라이더 값에 따라 이미지 업데이트"""
    frame = picam2.capture_array()
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

    lower_hsv = np.array([
        lower_h_slider.value,
        lower_s_slider.value,
        lower_v_slider.value
    ])
    upper_hsv = np.array([
        upper_h_slider.value,
        upper_s_slider.value,
        upper_v_slider.value
    ])

    # 윤곽선 찾기
    original_frame, color_mask, result_frame = find_contours_in_color_range(frame, lower_hsv, upper_hsv)

    # 이미지를 JPEG로 변환
    original_bytes = convert_to_bytes(original_frame)
    mask_bytes = convert_to_bytes(color_mask)
    result_bytes = convert_to_bytes(result_frame)
    
    # 원본 이미지와 결과 이미지 출력
    original_image_widget.value = original_bytes
    mask_image_widget.value = mask_bytes
    result_image_widget.value = result_bytes

def print_hsv_values(button):
    """버튼 클릭 시 현재 HSV 값 출력"""
    l_h = lower_h_slider.value
    l_s = lower_s_slider.value
    l_v = lower_v_slider.value
    u_h = upper_h_slider.value
    u_s = upper_s_slider.value
    u_v = upper_v_slider.value
    
    print(f"Lower HSV: ({l_h}, {l_s}, {l_v})")
    print(f"Upper HSV: ({u_h}, {u_s}, {u_v})")

# 슬라이더와 버튼 생성
lower_h_slider = widgets.IntSlider(value=0, min=0, max=179, step=1, description='Lower H')
lower_s_slider = widgets.IntSlider(value=0, min=0, max=255, step=1, description='Lower S')
lower_v_slider = widgets.IntSlider(value=0, min=0, max=255, step=1, description='Lower V')
upper_h_slider = widgets.IntSlider(value=179, min=0, max=179, step=1, description='Upper H')
upper_s_slider = widgets.IntSlider(value=255, min=0, max=255, step=1, description='Upper S')
upper_v_slider = widgets.IntSlider(value=255, min=0, max=255, step=1, description='Upper V')

print_button = widgets.Button(description="Print HSV Values")
print_button.on_click(print_hsv_values)

# 위젯을 통해 비디오 스트림 출력
original_image_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='320px', height='180px'))
mask_image_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='320px', height='180px'))
result_image_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='320px', height='180px'))

# 위젯 디스플레이
display(lower_h_slider, lower_s_slider, lower_v_slider, upper_h_slider, upper_s_slider, upper_v_slider, print_button)
display(widgets.HBox([original_image_widget, mask_image_widget, result_image_widget]))

def video_stream():
    """비디오 스트림을 처리하고 위젯에 업데이트합니다."""
    while True:
        update_image()
        # 주기적으로 업데이트 (프레임 간 간격 조절)
        time.sleep(0.2)  # 목표 프레임 속도 5 FPS

# 비디오 스트림 처리 스레드 시작
threading.Thread(target=video_stream, daemon=True).start()
